# PFA + EPPO Adaptive Learning — Complete POC

Full pipeline in one notebook: **PFA knowledge tracer** → **EPPO recommender** → **evaluation**

### Architecture
```
RealisticStudent  ->  PFATracker (per-session)  ->  EPPOAgent  ->  action: (concept, Bloom level)
      ^                      ^                            |
      +----------------------+----------- reward <--------+
```

| Section | Content |
|---------|---------|
| 1 | Install deps |
| 2 | Imports & GPU check |
| 3 | **Config** — concepts, hyperparams |
| 4 | **PFATracker** — knowledge tracer |
| 5 | **RealisticStudent** — 3PL IRT simulator |
| 6 | **EPPOAgent** — actor-critic with Bloom mask |
| 7 | **Reward** — ALPN-style R1+R2+R3 |
| 8 | **Rollout buffer + GAE + PPO update** |
| 9 | **PFA validation** — 8-test suite |
| 10 | **Training loop** — warm-up then EPPO |
| 11 | **Run training** |
| 12 | **Training curves** |
| 13 | **Policy comparison** — EPPO vs random vs greedy |
| 14 | **Demo session** — step-by-step trace |
| 15 | **Readiness checklist** |

**Runtime on Kaggle T4:** ~25–30 min end-to-end.


## 1 · Install dependencies

In [ ]:
import subprocess
subprocess.run(["pip", "install", "sentence-transformers", "scikit-learn", "-q"], check=True)
print("Dependencies ready.")


## 2 · Imports & GPU check

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
from collections import defaultdict
import os
import warnings
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
matplotlib.rcParams["figure.figsize"] = (16, 4)

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")


## 3 · Config

All hyperparameters in one place. Edit here before running training.

| Group | Key params |
|-------|-----------|
| Concepts | 8 CE concepts, 6 Bloom levels → 48 actions |
| PFA | `GAMMA_LEVEL` (learning rate), `RHO_LEVEL` (penalty), `BETA_LEVEL` (difficulty prior) |
| Student | `SLIP=0.08`, `GUESS=0.15`, `LEARN_RATE=0.04`, `FORGET_RATE=0.005` |
| EPPO | `LR=3e-4`, `CLIP_EPS=0.2`, `ENTROPY_COEF=0.015` |
| Training | `N_EPISODES=1500`, `MAX_STEPS=35`, `BETA_APR=0.72` (goal) |


In [ ]:
class Config:
    # ── Concept set ──────────────────────────────────────────────────────────
    CONCEPTS = [
        "binary search tree",
        "linked list",
        "sorting algorithms",
        "dynamic programming",
        "process scheduling",
        "memory management",
        "TCP IP basics",
        "recursion",
    ]
    N_CONCEPTS   = len(CONCEPTS)
    BLOOM_LEVELS = 6                          # Bloom L1..L6
    N_ACTIONS    = N_CONCEPTS * BLOOM_LEVELS  # 48

    # ── PFA hyperparameters ───────────────────────────────────────────────────
    PFA_TOP_K   = 3     # similarity neighbours used for propagation
    PFA_ALPHA   = 0.05  # propagation strength

    # Difficulty priors per Bloom level (aligned to student IRT curve)
    BETA_LEVEL  = np.array([ 0.0, -0.4, -0.9, -1.5, -2.1, -2.9])
    # Learning-rate factor per level (higher = faster mastery gain)
    GAMMA_LEVEL = np.array([ 0.40,  0.36,  0.32,  0.28,  0.24,  0.20])
    # Failure-penalty factor per level
    RHO_LEVEL   = np.array([ 0.28,  0.33,  0.38,  0.43,  0.50,  0.58])

    # ── Student simulator ─────────────────────────────────────────────────────
    SLIP           = 0.08   # P(wrong | truly knows)
    GUESS          = 0.15   # P(correct | truly doesn't know)
    LEARN_RATE     = 0.04   # ability gain per correct answer
    FORGET_RATE    = 0.005  # ability decay per sqrt(idle steps)
    TRANSFER_ALPHA = 0.30   # concept-transfer fraction via similarity

    # ── EPPO ──────────────────────────────────────────────────────────────────
    STATE_DIM    = N_CONCEPTS * 4  # 4 features per concept
    HIDDEN_DIM   = 128
    LR_ACTOR     = 3e-4
    LR_CRITIC    = 3e-4
    GAMMA        = 0.99
    GAE_LAMBDA   = 0.95
    CLIP_EPS     = 0.20
    ENTROPY_COEF = 0.015
    VALUE_COEF   = 0.50
    PPO_EPOCHS   = 4
    GRAD_CLIP    = 0.50

    # ── Training ──────────────────────────────────────────────────────────────
    WARMUP_EPISODES  = 300   # random episodes before EPPO starts
    N_EPISODES       = 1500  # EPPO training episodes
    MAX_STEPS        = 35    # max interactions per session
    BETA_APR         = 0.72  # APR goal (average pass rate across concepts)
    MIN_COVERAGE     = True  # must visit every concept at least once
    MAX_SAME_CONCEPT = 3     # max consecutive repeats of same concept
    LOG_EVERY        = 100
    SAVE_EVERY       = 500

    # ── Reward weights ────────────────────────────────────────────────────────
    W_MASTERY = 1.0   # mastery-gain weight
    W_FIT     = 0.40  # difficulty-mismatch penalty
    W_DIV     = 0.08  # repetition-diversity penalty

    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


cfg = Config()
print(f"Concepts  : {cfg.N_CONCEPTS}")
print(f"Actions   : {cfg.N_ACTIONS}  ({cfg.N_CONCEPTS} concepts x {cfg.BLOOM_LEVELS} Bloom levels)")
print(f"State dim : {cfg.STATE_DIM}  ({cfg.N_CONCEPTS} x 4 features)")
print(f"Device    : {cfg.DEVICE}")
print(f"Goal APR  : {cfg.BETA_APR}")


## 4 · PFA Knowledge Tracer

**Performance Factor Analysis** extended with:
- Sentence-transformer similarity graph (built once, shared across all sessions)
- Per-concept, per-Bloom-level success/failure counts
- Neighbour propagation via a *separate* bonus array (no ghost counts)
- Fixed-size state vector for EPPO (4 features × N_concepts)

Key methods: `predict(concept, level)` → P(correct) · `update(concept, level, correct)` · `get_state_vector()`


In [ ]:
class PFATracker:
    \"\"\"
    PFA knowledge tracer with sentence-embedding similarity graph.

    predict(concept, level)      -> P(correct) in (0, 1)
    update(concept, level, bool) -> update counts + propagate to neighbours
    get_state_vector()           -> fixed-size np.float32 array for EPPO
    \"\"\"

    def __init__(self, concepts, cfg, sim_matrix=None):
        self.concepts      = concepts
        self.cfg           = cfg
        self.num_levels    = cfg.BLOOM_LEVELS
        self.top_k         = cfg.PFA_TOP_K
        self.alpha         = cfg.PFA_ALPHA
        self.num_concepts  = len(concepts)
        self.idx           = {c: i for i, c in enumerate(concepts)}

        # Real interaction counts — only update() writes here
        self.successes         = np.zeros((self.num_concepts, self.num_levels))
        self.failures          = np.zeros((self.num_concepts, self.num_levels))
        # Propagation bonus — separate array, never mixed with raw counts
        self.propagation_bonus = np.zeros((self.num_concepts, self.num_levels))
        # Track attempted concepts (for state vector feature)
        self.attempted         = np.zeros(self.num_concepts, dtype=bool)

        # Concept-level bias (fixed seed -> deterministic across all instances)
        rng_prior         = np.random.default_rng(42)
        self.beta_concept = rng_prior.uniform(-0.3, 0.3, self.num_concepts)

        # Similarity graph
        if sim_matrix is not None:
            self.sim_matrix = sim_matrix
            self._build_neighbors()
        else:
            self._build_similarity_graph()

    # ── Graph ────────────────────────────────────────────────────────────────

    def _build_similarity_graph(self):
        print("  Building similarity graph via sentence-transformers...")
        from sentence_transformers import SentenceTransformer
        model = SentenceTransformer("BAAI/bge-base-en-v1.5")
        emb = model.encode(self.concepts, normalize_embeddings=True)
        self.sim_matrix = emb @ emb.T
        self._build_neighbors()
        print(f"  Done. Top-3 neighbours of '{self.concepts[0]}':")
        for j in self.neighbors[0]:
            print(f"    {self.concepts[j]}  sim={self.sim_matrix[0, j]:.3f}")

    def _build_neighbors(self):
        self.neighbors = []
        for i in range(self.num_concepts):
            ranked = np.argsort(-self.sim_matrix[i])
            self.neighbors.append([j for j in ranked if j != i][: self.top_k])

    # ── Prediction ───────────────────────────────────────────────────────────

    def predict(self, concept, level):
        \"\"\"Return P(correct) for concept at Bloom level (1-indexed).\"\"\"
        i, k = self.idx[concept], level - 1
        cfg  = self.cfg
        z = (
            self.beta_concept[i]
            + cfg.BETA_LEVEL[k]
            + cfg.GAMMA_LEVEL[k] * np.log1p(self.successes[i, k])
            - cfg.RHO_LEVEL[k]   * np.log1p(self.failures[i, k])
            + self.propagation_bonus[i, k]
        )
        if k > 0:  # lower-level scaffold
            z += 0.25 * (
                cfg.GAMMA_LEVEL[k-1] * np.log1p(self.successes[i, k-1])
                - cfg.RHO_LEVEL[k-1] * np.log1p(self.failures[i, k-1])
            )
        return float(1 / (1 + np.exp(-z)))

    def predict_idx(self, ci, level):
        return self.predict(self.concepts[ci], level)

    # ── Update ───────────────────────────────────────────────────────────────

    def update(self, concept, level, correct):
        \"\"\"Record one interaction and propagate delta to top-k neighbours.\"\"\"
        i, k = self.idx[concept], level - 1
        self.attempted[i] = True
        old_p = self.predict(concept, level)
        if correct:
            self.successes[i, k] += 1
        else:
            self.failures[i, k]  += 1
        delta = np.clip(self.predict(concept, level) - old_p, -0.10, 0.10)
        for j in self.neighbors[i]:
            sim = self.sim_matrix[i, j]
            for lvl in range(k + 1):
                self.propagation_bonus[j, lvl] += self.alpha * sim / (lvl + 1) * delta

    def update_idx(self, ci, level, correct):
        self.update(self.concepts[ci], level, correct)

    # ── State vector for EPPO ────────────────────────────────────────────────

    def get_state_vector(self):
        \"\"\"
        4 features per concept -> shape (N_CONCEPTS * 4,).
          [0] mean mastery across all Bloom levels
          [1] highest level where P > 0.60, normalised to [0, 1]
          [2] variance across Bloom levels
          [3] 1 if concept was attempted in this session, else 0
        \"\"\"
        feats = []
        for i, c in enumerate(self.concepts):
            probs    = np.array([self.predict(c, l + 1) for l in range(self.num_levels)])
            mean_m   = float(probs.mean())
            max_lvl  = max((l for l in range(self.num_levels) if probs[l] > 0.60), default=-1)
            norm_lvl = (max_lvl + 1) / self.num_levels
            variance = float(probs.var())
            feats.extend([mean_m, norm_lvl, variance, float(self.attempted[i])])
        return np.array(feats, dtype=np.float32)

    def get_mastery_matrix(self):
        \"\"\"Full (N_concepts, N_levels) mastery probability matrix.\"\"\"
        M = np.zeros((self.num_concepts, self.num_levels))
        for i, c in enumerate(self.concepts):
            for k in range(self.num_levels):
                M[i, k] = self.predict(c, k + 1)
        return M

    def reset(self):
        self.successes[:]          = 0
        self.failures[:]           = 0
        self.propagation_bonus[:]  = 0
        self.attempted[:]          = False


print("PFATracker defined.")


## 5 · Realistic Student Simulator

**3PL IRT** student with:
- Per-concept × per-Bloom-level ability matrix `(C, 6)`
- `answer()` → slip/guess-adjusted correct probability
- `learn()` → ability gain with cross-level scaffold + concept transfer via sim_matrix
- `_apply_forgetting()` → sqrt(elapsed) decay
- `from_archetype()` → beginner / intermediate / advanced / mixed prior


In [ ]:
class RealisticStudent:
    \"\"\"
    3PL IRT student: slip, guess, per-concept x per-level ability,
    forgetting, and concept transfer via the PFA similarity matrix.
    \"\"\"

    def __init__(self, concepts, sim_matrix, rng, cfg):
        self.concepts       = concepts
        self.sim_matrix     = sim_matrix
        self.rng            = rng
        self.slip           = cfg.SLIP
        self.guess          = cfg.GUESS
        self.learn_rate     = cfg.LEARN_RATE
        self.forget_rate    = cfg.FORGET_RATE
        self.transfer_alpha = cfg.TRANSFER_ALPHA

        num_c        = len(concepts)
        base         = rng.uniform(-1.5, 0.5, size=num_c)
        level_offset = np.array([1.0, 0.6, 0.2, -0.2, -0.7, -1.3])
        self.ability         = base[:, None] + level_offset[None, :]  # (C, 6)
        self.item_difficulty = np.array([0.0, 0.5, 1.0, 1.5, 2.2, 3.0])
        self.last_attempt    = defaultdict(lambda: 0)
        self.t               = 0

    @classmethod
    def from_archetype(cls, concepts, sim_matrix, rng, cfg, archetype="mixed"):
        \"\"\"Construct a student with a specific prior ability level.\"\"\"
        s = cls(concepts, sim_matrix, rng, cfg)
        num_c = len(concepts)
        ranges = {
            "beginner":     (-1.8, -0.5),
            "intermediate": (-0.8,  0.3),
            "advanced":     (-0.2,  1.0),
            "mixed":        (-1.5,  0.5),
        }
        lo, hi = ranges.get(archetype, (-1.5, 0.5))
        base   = rng.uniform(lo, hi, size=num_c)
        level_offset = np.array([1.0, 0.6, 0.2, -0.2, -0.7, -1.3])
        s.ability = base[:, None] + level_offset[None, :]
        return s

    def _apply_forgetting(self, ci, k):
        elapsed = self.t - self.last_attempt[(ci, k)]
        if elapsed > 0:
            self.ability[ci, k] = max(
                self.ability[ci, k] - self.forget_rate * np.sqrt(elapsed), -3.0)

    def answer(self, ci, level):
        \"\"\"
        Simulate answering a question.
        Returns (correct: bool, true_p: float, p_know: float).
        level is 1-indexed.
        \"\"\"
        self.t += 1
        k = level - 1
        self._apply_forgetting(ci, k)
        logit  = self.ability[ci, k] - self.item_difficulty[k]
        p_know = float(1 / (1 + np.exp(-logit)))
        true_p = self.guess + (1 - self.slip - self.guess) * p_know
        correct = bool(self.rng.random() < true_p)
        self.last_attempt[(ci, k)] = self.t
        return correct, true_p, p_know

    def learn(self, ci, level, correct):
        \"\"\"Update ability after an interaction.\"\"\"
        k = level - 1
        if correct:
            self.ability[ci, k] += self.learn_rate
            if k > 0:                             # boost lower level slightly
                self.ability[ci, k-1] += self.learn_rate * 0.3
        else:
            self.ability[ci, k] = max(self.ability[ci, k] - self.learn_rate * 0.3, -3.0)
        # Concept transfer to similar neighbours
        for j, sim in enumerate(self.sim_matrix[ci]):
            if j == ci or sim < 0.50:
                continue
            self._apply_forgetting(j, k)
            if correct:
                self.ability[j, k] += self.transfer_alpha * sim * self.learn_rate


print("RealisticStudent defined.")


## 6 · EPPO Agent

**Entropy-enhanced PPO** actor-critic.

- **State**: 32-dim fixed vector from PFA (4 features × 8 concepts)
- **Action**: integer in [0, 48) → decoded as `(concept_idx, bloom_level)`
- **Action mask**: competence ceiling — for each concept, allows current mastery level + one above; L1 always unmasked
- **EPPO twist**: entropy bonus uses the *stored* rollout entropy rather than freshly computed entropy, preserving the early exploration signal across PPO epochs


In [ ]:
class EPPOAgent(nn.Module):
    \"\"\"
    Entropy-enhanced PPO actor-critic.

    Action space: concept_idx * BLOOM_LEVELS + (level - 1)  ->  int in [0, N_ACTIONS)
    \"\"\"

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.actor = nn.Sequential(
            nn.Linear(cfg.STATE_DIM, cfg.HIDDEN_DIM), nn.ReLU(),
            nn.Linear(cfg.HIDDEN_DIM, cfg.HIDDEN_DIM), nn.ReLU(),
            nn.Linear(cfg.HIDDEN_DIM, cfg.N_ACTIONS),
        )
        self.critic = nn.Sequential(
            nn.Linear(cfg.STATE_DIM, cfg.HIDDEN_DIM), nn.ReLU(),
            nn.Linear(cfg.HIDDEN_DIM, cfg.HIDDEN_DIM), nn.ReLU(),
            nn.Linear(cfg.HIDDEN_DIM, 1),
        )

    def get_action_mask(self, mastery_matrix):
        \"\"\"
        Competence-ceiling mask.
        For each concept: allow levels up to (current_mastery_level + 1).
        L1 (Recall) is always allowed as a fallback.
        \"\"\"
        cfg  = self.cfg
        mask = torch.zeros(cfg.N_ACTIONS, dtype=torch.bool)
        for c in range(cfg.N_CONCEPTS):
            current = max(
                (k for k in range(cfg.BLOOM_LEVELS) if mastery_matrix[c, k] > 0.55),
                default=0,
            )
            for k in range(cfg.BLOOM_LEVELS):
                if k <= current + 1:
                    mask[c * cfg.BLOOM_LEVELS + k] = True
            mask[c * cfg.BLOOM_LEVELS] = True  # L1 always available
        return mask

    def select_action(self, state, mastery_matrix):
        \"\"\"Sample an action at inference / rollout time.\"\"\"
        state_t = torch.FloatTensor(state).to(self.cfg.DEVICE)
        logits  = self.actor(state_t)
        mask    = self.get_action_mask(mastery_matrix).to(self.cfg.DEVICE)
        logits  = logits.masked_fill(~mask, float("-inf"))
        dist    = Categorical(logits=logits)
        action  = dist.sample()
        return action.item(), dist.log_prob(action).item(), dist.entropy().item(), self.critic(state_t).item()

    def evaluate(self, states, actions):
        \"\"\"Evaluate a batch of (state, action) pairs for the PPO update.\"\"\"
        logits    = self.actor(states).clamp(min=-1e9)
        values    = self.critic(states).squeeze(1)
        dist      = Categorical(logits=logits)
        return dist.log_prob(actions), values, dist.entropy()


print("EPPOAgent defined.")


## 7 · Reward function (ALPN-inspired)

Three additive terms:

| Term | Description | Formula |
|------|-------------|---------|
| **R1** mastery gain | APR delta, scaled by remaining distance to goal | `W_MASTERY * (gain * C) / dist_to_goal` |
| **R2** difficulty fit | Penalise Bloom level mismatch from ideal | `-W_FIT * mismatch * dir_weight` |
| **R3** diversity | Penalise repeating exact (concept, level) action | `-W_DIV * n_repeats` |

Being too easy is penalised harder than too hard (dir_weight 1.2 vs 0.6) to encourage the agent to challenge students upward.


In [ ]:
def compute_reward(mastery_before, mastery_after, ci, level, correct, action_history, cfg):
    \"\"\"
    ALPN-style three-term reward.

    mastery_before / mastery_after : np.ndarray (N_concepts, N_levels)
    ci     : concept index (int)
    level  : Bloom level 1-indexed (int)
    correct: whether student answered correctly (bool)
    action_history: list of previous action ints this episode
    \"\"\"
    k = level - 1
    C = cfg.N_CONCEPTS

    # R1 — mastery gain (APR = mean L1 mastery across all concepts)
    apr_before   = float(np.mean(mastery_before[:, 0]))
    apr_after    = float(np.mean(mastery_after[:, 0]))
    gain         = apr_after - apr_before
    dist_to_goal = max(cfg.BETA_APR - apr_after, 1e-6)
    R1 = float(np.clip(
        cfg.W_MASTERY * (gain * C) / dist_to_goal if correct else cfg.W_MASTERY * gain * C,
        -5.0, 10.0,
    ))

    # R2 — difficulty fit
    current_k = max((kk for kk in range(cfg.BLOOM_LEVELS) if mastery_before[ci, kk] > 0.55), default=0)
    ideal_k   = min(current_k + 1, cfg.BLOOM_LEVELS - 1)
    mismatch  = abs(k - ideal_k)
    dir_w     = 1.2 if k < ideal_k else 0.6   # too easy penalised harder
    R2 = -cfg.W_FIT * mismatch * dir_w

    # R3 — diversity (only on correct answers — repeating failures is less wasteful)
    action = ci * cfg.BLOOM_LEVELS + k
    R3 = -cfg.W_DIV * action_history.count(action) if correct else 0.0

    return float(R1 + R2 + R3)


print("compute_reward defined.")


## 8 · Rollout buffer · GAE · PPO update

Standard on-policy rollout with Generalised Advantage Estimation.  
`ppo_update` runs `PPO_EPOCHS` mini-epochs with clipped surrogate + value loss.  
Entropy bonus uses the **stored** per-step entropy (EPPO behaviour).


In [ ]:
class RolloutBuffer:
    def __init__(self):
        self.states = []; self.actions   = []
        self.log_probs = []; self.rewards = []
        self.values = []; self.entropies  = []
        self.dones  = []

    def store(self, s, a, lp, r, v, e, d):
        self.states.append(s);    self.actions.append(a)
        self.log_probs.append(lp); self.rewards.append(r)
        self.values.append(v);    self.entropies.append(e)
        self.dones.append(d)

    def clear(self): self.__init__()
    def __len__(self): return len(self.states)

    def get_tensors(self, device):
        return (
            torch.FloatTensor(np.array(self.states)).to(device),
            torch.LongTensor(self.actions).to(device),
            torch.FloatTensor(self.log_probs).to(device),
            torch.FloatTensor(self.rewards).to(device),
            torch.FloatTensor(self.values).to(device),
            torch.FloatTensor(self.entropies).to(device),
            torch.FloatTensor(self.dones).to(device),
        )


def compute_gae(rewards, values, dones, gamma, lam):
    advantages = []
    gae = 0.0
    next_val = 0.0
    for t in reversed(range(len(rewards))):
        delta = rewards[t] + gamma * next_val * (1 - dones[t]) - values[t]
        gae   = delta + gamma * lam * (1 - dones[t]) * gae
        advantages.insert(0, gae)
        next_val = values[t]
    advantages = torch.FloatTensor(advantages)
    returns    = advantages + torch.FloatTensor(values)
    return advantages, returns


def ppo_update(agent, buffer, actor_opt, critic_opt, cfg):
    states, actions, old_lp, rewards, values, stored_ent, dones = buffer.get_tensors(cfg.DEVICE)

    advantages, returns = compute_gae(
        rewards.cpu().numpy(), values.cpu().numpy(),
        dones.cpu().numpy(), cfg.GAMMA, cfg.GAE_LAMBDA,
    )
    advantages = advantages.to(cfg.DEVICE)
    returns    = returns.to(cfg.DEVICE)
    if advantages.numel() > 1:
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

    total_al = total_cl = 0.0
    for _ in range(cfg.PPO_EPOCHS):
        new_lp, new_vals, _ = agent.evaluate(states, actions)
        ratio  = torch.exp(new_lp - old_lp.detach())
        surr1  = ratio * advantages
        surr2  = torch.clamp(ratio, 1 - cfg.CLIP_EPS, 1 + cfg.CLIP_EPS) * advantages
        a_loss = -torch.min(surr1, surr2).mean()
        c_loss = nn.MSELoss()(new_vals, returns)
        # EPPO: use stored entropy to preserve early exploration signal
        loss   = a_loss + cfg.VALUE_COEF * c_loss - cfg.ENTROPY_COEF * stored_ent.mean()

        actor_opt.zero_grad(); critic_opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(agent.parameters(), cfg.GRAD_CLIP)
        actor_opt.step(); critic_opt.step()

        total_al += a_loss.item()
        total_cl += c_loss.item()

    n = cfg.PPO_EPOCHS
    return total_al / n, total_cl / n


print("RolloutBuffer, compute_gae, ppo_update defined.")


## 9 · PFA validation — 8-test suite

Validates the tracker in isolation **before** attaching EPPO.

| Test | What it checks | Pass condition |
|------|---------------|----------------|
| T1 | Always-correct → monotone rise | 0 drops, gain > 0.05 |
| T2 | Always-wrong → monotone drop | 0 rises, drop > 0.05 |
| T3 | Bloom ordering L1 > L2 > … > L6 | Monotone decreasing |
| T4 | Calibration ECE | ECE < 0.10 |
| T5 | Steps to 80% mastery | 10–70 steps |
| T6 | Propagation ordering | Near neighbour moves more than far |
| T7 | Ghost counts | Raw counts untouched for non-attempted concepts |
| T8 | Recovery after failure | Recovery ratio > 70% |


In [ ]:
def run_pfa_validation(cfg, sim_matrix, rng):
    from sklearn.metrics import roc_auc_score

    print("\n" + "=" * 55)
    print("  PFA VALIDATION (8 tests)")
    print("=" * 55)

    probe   = cfg.CONCEPTS[0]
    results = {}

    def fresh():
        return PFATracker(cfg.CONCEPTS, cfg, sim_matrix=sim_matrix)

    def make_s():
        return RealisticStudent(cfg.CONCEPTS, sim_matrix, rng, cfg)

    # T1 — always correct
    t, s = fresh(), make_s()
    preds = []
    for _ in range(60):
        preds.append(t.predict(probe, 1))
        t.update(probe, 1, True); s.learn(t.idx[probe], 1, True)
    drops = sum(1 for a, b in zip(preds, preds[1:]) if b < a - 1e-9)
    gain  = preds[-1] - preds[0]
    t1_ok = drops == 0 and gain > 0.05 and preds[-1] <= 1.0
    print(f"  T1 always-correct:   start={preds[0]:.3f} end={preds[-1]:.3f} gain={gain:+.3f}  {'PASS' if t1_ok else 'FAIL'}")
    results["t1"] = t1_ok

    # T2 — always wrong
    t, s = fresh(), make_s()
    preds = []
    for _ in range(60):
        preds.append(t.predict(probe, 1))
        t.update(probe, 1, False); s.learn(t.idx[probe], 1, False)
    rises = sum(1 for a, b in zip(preds, preds[1:]) if b > a + 1e-9)
    drop  = preds[0] - preds[-1]
    t2_ok = rises == 0 and drop > 0.05 and preds[-1] >= 0.0
    print(f"  T2 always-wrong:     start={preds[0]:.3f} end={preds[-1]:.3f} drop={drop:+.3f}  {'PASS' if t2_ok else 'FAIL'}")
    results["t2"] = t2_ok

    # T3 — bloom ordering
    t, s = fresh(), make_s()
    ci = t.idx[probe]
    for k in range(cfg.BLOOM_LEVELS):
        for _ in range(40):
            t.update(probe, k + 1, True); s.learn(ci, k + 1, True)
    fp = [t.predict(probe, l + 1) for l in range(cfg.BLOOM_LEVELS)]
    t3_ok = all(a >= b for a, b in zip(fp, fp[1:]))
    print(f"  T3 bloom ordering:   {[f'{p:.2f}' for p in fp]}  {'PASS' if t3_ok else 'FAIL'}")
    results["t3"] = t3_ok

    # T4 — calibration
    t, s = fresh(), make_s()
    y_true, y_pred = [], []
    for _ in range(1500):
        c_name = rng.choice(cfg.CONCEPTS)
        ci     = t.idx[c_name]
        lvl    = int(rng.integers(1, cfg.BLOOM_LEVELS + 1))
        correct, _, _ = s.answer(ci, lvl)
        y_pred.append(t.predict(c_name, lvl))
        y_true.append(int(correct))
        t.update(c_name, lvl, correct); s.learn(ci, lvl, correct)
    y_true = np.array(y_true); y_pred = np.array(y_pred)
    ece = sum(
        (((y_pred >= i/10) & (y_pred < (i+1)/10)).sum() / len(y_pred))
        * abs(y_true[(y_pred >= i/10) & (y_pred < (i+1)/10)].mean() - y_pred[(y_pred >= i/10) & (y_pred < (i+1)/10)].mean())
        for i in range(10)
        if ((y_pred >= i/10) & (y_pred < (i+1)/10)).sum() > 0
    )
    auc   = roc_auc_score(y_true, y_pred)
    t4_ok = ece < 0.10
    print(f"  T4 calibration:      ECE={ece:.4f} AUC={auc:.3f}  {'PASS' if t4_ok else 'FAIL (ECE>=0.10)'}")
    results.update({"t4": t4_ok, "ece": ece, "auc": auc})

    # T5 — steps to mastery
    t, s = fresh(), make_s()
    ci = t.idx[probe]; crossed = None
    for step in range(200):
        if t.predict(probe, 1) >= 0.80:
            crossed = step; break
        t.update(probe, 1, True); s.learn(ci, 1, True)
    t5_ok = crossed is not None and 10 <= crossed <= 70
    print(f"  T5 steps-to-mastery: crossed 80% at step {crossed}  {'PASS' if t5_ok else 'FAIL'}")
    results["t5"] = t5_ok

    # T6 — propagation ordering
    t, s = fresh(), make_s()
    ci = t.idx[probe]
    ranked     = [j for j in np.argsort(-t.sim_matrix[ci]) if j != ci]
    near, far  = ranked[0], ranked[-1]
    b_near     = t.predict_idx(near, 1)
    b_far      = t.predict_idx(far,  1)
    for _ in range(30):
        t.update(probe, 1, True); s.learn(ci, 1, True)
    d_near  = t.predict_idx(near, 1) - b_near
    d_far   = t.predict_idx(far,  1) - b_far
    ghost_ok = t.successes[near, 0] == 0 and t.failures[near, 0] == 0
    t6_ok    = d_near > d_far and ghost_ok
    print(f"  T6 propagation:      Δnear={d_near:+.4f} Δfar={d_far:+.4f} ghost_ok={ghost_ok}  {'PASS' if t6_ok else 'FAIL'}")
    results["t6"] = t6_ok

    # T7 — ghost counts
    t, s   = fresh(), make_s()
    ci     = t.idx[probe]
    others = [c for c in cfg.CONCEPTS if c != probe]
    b_succ = {c: t.successes[t.idx[c]].copy() for c in others}
    b_fail = {c: t.failures[t.idx[c]].copy()  for c in others}
    for _ in range(50):
        t.update(probe, 1, True); s.learn(ci, 1, True)
    ghost = any(
        not (np.allclose(b_succ[c], t.successes[t.idx[c]]) and
             np.allclose(b_fail[c], t.failures[t.idx[c]]))
        for c in others
    )
    t7_ok = not ghost
    print(f"  T7 ghost counts:     ghost_found={ghost}  {'PASS' if t7_ok else 'FAIL'}")
    results["t7"] = t7_ok

    # T8 — recovery
    t, s = fresh(), make_s()
    ci = t.idx[probe]
    for _ in range(25): t.update(probe, 1, True);  s.learn(ci, 1, True)
    peak = t.predict(probe, 1)
    for _ in range(25): t.update(probe, 1, False); s.learn(ci, 1, False)
    trough = t.predict(probe, 1)
    for _ in range(25): t.update(probe, 1, True);  s.learn(ci, 1, True)
    recovery = t.predict(probe, 1)
    ratio = (recovery - trough) / max(peak - trough, 1e-6)
    t8_ok = ratio > 0.70
    print(f"  T8 recovery:         peak={peak:.3f} trough={trough:.3f} recovery={recovery:.3f} ratio={ratio:.1%}  {'PASS' if t8_ok else 'FAIL'}")
    results["t8"] = t8_ok

    n_pass = sum(results[k] for k in ["t1","t2","t3","t4","t5","t6","t7","t8"])
    print(f"\n  Result: {n_pass}/8 passed  |  ECE={results['ece']:.4f}  AUC={results['auc']:.3f}")
    print("=" * 55)
    return results


print("run_pfa_validation defined.")


## 10 · Training loop

Two phases:

**Phase 1 — warm-up** (`WARMUP_EPISODES=300`): Random interactions to confirm PFA handles diverse trajectories without instability. PFA has no learnable weights, so this is a stability sanity-check only.

**Phase 2 — EPPO** (`N_EPISODES=1500`): Each episode spawns a *fresh* PFA tracker and a *fresh* RealisticStudent (sampled from a random archetype). The agent observes the state, picks an action, the student answers, the tracker updates, reward is computed, buffer is stored, PPO update runs end-of-episode.

Logs every 100 episodes. Saves a checkpoint every 500.


In [ ]:
def train(cfg, save_dir="checkpoints"):
    os.makedirs(save_dir, exist_ok=True)
    rng = np.random.default_rng(42)

    # ── Build similarity graph once ─────────────────────────────────────────
    print("Building PFA similarity graph...")
    dummy   = PFATracker(cfg.CONCEPTS, cfg)
    sim_matrix = dummy.sim_matrix

    # ── Phase 0: PFA validation ─────────────────────────────────────────────
    val_results = run_pfa_validation(cfg, sim_matrix, np.random.default_rng(99))
    if val_results.get("ece", 1.0) >= 0.15:
        print("WARNING: ECE >= 0.15. Tune GAMMA_LEVEL / RHO_LEVEL before scaling up.")

    # ── Phase 1: Warm-up ────────────────────────────────────────────────────
    print(f"\nPhase 1 — warm-up ({cfg.WARMUP_EPISODES} random episodes)...")
    for _ in range(cfg.WARMUP_EPISODES):
        t = PFATracker(cfg.CONCEPTS, cfg, sim_matrix=sim_matrix)
        s = RealisticStudent.from_archetype(cfg.CONCEPTS, sim_matrix, rng, cfg, "mixed")
        for _ in range(cfg.MAX_STEPS):
            ci  = int(rng.integers(0, cfg.N_CONCEPTS))
            lvl = int(rng.integers(1, cfg.BLOOM_LEVELS + 1))
            correct, _, _ = s.answer(ci, lvl)
            t.update_idx(ci, lvl, correct); s.learn(ci, lvl, correct)
    print("  Warm-up complete.")

    # ── Phase 2: EPPO training ──────────────────────────────────────────────
    print(f"\nPhase 2 — EPPO training ({cfg.N_EPISODES} episodes) on {cfg.DEVICE}")
    print(f"  State={cfg.STATE_DIM}  Actions={cfg.N_ACTIONS}  Goal APR={cfg.BETA_APR}")
    print("-" * 58)

    agent      = EPPOAgent(cfg).to(cfg.DEVICE)
    buffer     = RolloutBuffer()
    actor_opt  = optim.Adam(agent.actor.parameters(),  lr=cfg.LR_ACTOR)
    critic_opt = optim.Adam(agent.critic.parameters(), lr=cfg.LR_CRITIC)

    metrics = {"rewards": [], "aprs": [], "steps": [], "goals": [],
               "actor_loss": [], "critic_loss": []}

    for episode in range(cfg.N_EPISODES):
        tracker   = PFATracker(cfg.CONCEPTS, cfg, sim_matrix=sim_matrix)
        archetype = rng.choice(["beginner", "intermediate", "advanced", "mixed"])
        student   = RealisticStudent.from_archetype(cfg.CONCEPTS, sim_matrix, rng, cfg, archetype)

        buffer.clear()
        action_hist      = []
        ep_reward        = 0.0
        concepts_covered = set()
        same_streak      = 0
        last_ci          = -1

        for step in range(cfg.MAX_STEPS):
            mastery = tracker.get_mastery_matrix()
            state   = tracker.get_state_vector()
            apr     = float(np.mean(mastery[:, 0]))

            all_covered   = len(concepts_covered) >= cfg.N_CONCEPTS
            can_terminate = (not cfg.MIN_COVERAGE) or all_covered
            if apr >= cfg.BETA_APR and can_terminate:
                break

            action, log_prob, entropy, value = agent.select_action(state, mastery)
            ci  = action // cfg.BLOOM_LEVELS
            lvl = action  % cfg.BLOOM_LEVELS + 1

            # Consecutive-concept guard
            same_streak = same_streak + 1 if ci == last_ci else 1
            last_ci     = ci
            if same_streak > cfg.MAX_SAME_CONCEPT:
                uncovered = [c for c in range(cfg.N_CONCEPTS) if c not in concepts_covered]
                ci        = int(rng.choice(uncovered)) if uncovered else int(np.argmin(mastery[:, 0]))
                action    = ci * cfg.BLOOM_LEVELS + (lvl - 1)
                same_streak = 1; last_ci = ci

            action_hist.append(action)
            concepts_covered.add(ci)

            mastery_before            = mastery.copy()
            correct, true_p, p_know   = student.answer(ci, lvl)
            tracker.update_idx(ci, lvl, correct)
            student.learn(ci, lvl, correct)
            mastery_after = tracker.get_mastery_matrix()

            reward    = compute_reward(mastery_before, mastery_after, ci, lvl, correct, action_hist, cfg)
            ep_reward += reward

            done = (step == cfg.MAX_STEPS - 1) or (
                float(np.mean(mastery_after[:, 0])) >= cfg.BETA_APR and can_terminate
            )
            buffer.store(state, action, log_prob, reward, value, entropy, float(done))
            if done:
                break

        # PPO update
        if len(buffer) > 0:
            al, cl = ppo_update(agent, buffer, actor_opt, critic_opt, cfg)
        else:
            al = cl = 0.0

        final_apr    = float(np.mean(tracker.get_mastery_matrix()[:, 0]))
        goal_reached = final_apr >= cfg.BETA_APR

        metrics["rewards"].append(ep_reward)
        metrics["aprs"].append(final_apr)
        metrics["steps"].append(len(action_hist))
        metrics["goals"].append(goal_reached)
        metrics["actor_loss"].append(al)
        metrics["critic_loss"].append(cl)

        if (episode + 1) % cfg.LOG_EVERY == 0:
            sl = slice(-cfg.LOG_EVERY, None)
            print(
                f"  Ep {episode+1:4d} | "
                f"Reward={np.mean(metrics['rewards'][sl]):+.2f} | "
                f"APR={np.mean(metrics['aprs'][sl]):.3f} | "
                f"Goal%={np.mean(metrics['goals'][sl])*100:.1f}% | "
                f"Steps={np.mean(metrics['steps'][sl]):.1f} | "
                f"Covered={len(concepts_covered)}/{cfg.N_CONCEPTS}"
            )

        if (episode + 1) % cfg.SAVE_EVERY == 0:
            torch.save(
                {"agent": agent.state_dict(), "episode": episode + 1, "metrics": metrics},
                os.path.join(save_dir, f"ckpt_{episode+1}.pt"),
            )

    torch.save(
        {"agent": agent.state_dict(), "episode": cfg.N_EPISODES, "metrics": metrics},
        os.path.join(save_dir, "final.pt"),
    )
    print("\nTraining complete.")
    return agent, sim_matrix, metrics, val_results


print("train() defined.")


## 11 · Run training

Expected: ~25–30 min on Kaggle T4.  
Watch `APR` trend upward and `Goal%` exceed 40% by episode ~800.


In [ ]:
agent, sim_matrix, metrics, val_results = train(cfg, save_dir="/kaggle/working/checkpoints")


## 12 · Training curves

What healthy training looks like:
- **APR**: steady climb toward 0.72, no sudden collapse
- **Reward**: positive trend, typical range −20 to +50
- **Goal %**: >50% by episode 700, ideally >65% by end
- **Losses**: actor stabilises, critic decreases


In [ ]:
def smooth(x, w=50):
    if len(x) < w:
        return np.array(x)
    return np.convolve(x, np.ones(w) / w, mode="valid")

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

axes[0].plot(smooth(metrics["aprs"]), color="steelblue", lw=2)
axes[0].axhline(cfg.BETA_APR, color="red", ls="--", alpha=0.7, label=f"goal={cfg.BETA_APR}")
axes[0].set_title("Average pass rate (APR)"); axes[0].set_ylim(0, 1)
axes[0].set_xlabel("Episode"); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(smooth(metrics["rewards"]), color="darkorange", lw=2)
axes[1].set_title("Episode reward"); axes[1].set_xlabel("Episode"); axes[1].grid(alpha=0.3)

axes[2].plot(smooth([float(g) for g in metrics["goals"]]) * 100, color="seagreen", lw=2)
axes[2].set_title("Goal reached %"); axes[2].set_ylim(0, 100)
axes[2].set_xlabel("Episode"); axes[2].grid(alpha=0.3)

axes[3].plot(smooth(metrics["actor_loss"]),  label="actor",  lw=1.5)
axes[3].plot(smooth(metrics["critic_loss"]), label="critic", lw=1.5)
axes[3].set_title("PPO losses"); axes[3].set_xlabel("Episode")
axes[3].legend(); axes[3].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("/kaggle/working/training_curves.png", dpi=120)
plt.show()

final_n = -300
print(f"Final {abs(final_n)} episodes:")
print(f"  Mean APR   : {np.mean(metrics['aprs'][final_n:]):.3f}")
print(f"  Goal rate  : {np.mean(metrics['goals'][final_n:])*100:.1f}%")
print(f"  Mean steps : {np.mean(metrics['steps'][final_n:]):.1f}")
print(f"  Mean reward: {np.mean(metrics['rewards'][final_n:]):+.2f}")


## 13 · Policy comparison

Runs 300 fresh students through each policy and reports mean APR, goal rate, and mean steps.

- **Random**: uniform random (concept, Bloom level)
- **Greedy**: always pick weakest concept at its ideal Bloom level
- **EPPO**: trained agent

EPPO must beat Random to demonstrate learning.  
EPPO matching Greedy in fewer steps counts as a win.


In [ ]:
def _run_policy_eval(policy_fn, cfg, sim_matrix, rng, n=300):
    aprs, steps, goals = [], [], []
    diff_counts = np.zeros(cfg.BLOOM_LEVELS)
    for _ in range(n):
        t = PFATracker(cfg.CONCEPTS, cfg, sim_matrix=sim_matrix)
        s = RealisticStudent.from_archetype(cfg.CONCEPTS, sim_matrix, rng, cfg, "mixed")
        covered = set()
        for step in range(cfg.MAX_STEPS):
            mastery = t.get_mastery_matrix()
            apr     = float(np.mean(mastery[:, 0]))
            if apr >= cfg.BETA_APR and ((not cfg.MIN_COVERAGE) or len(covered) >= cfg.N_CONCEPTS):
                break
            ci, lvl = policy_fn(t, mastery, rng, cfg)
            diff_counts[lvl - 1] += 1
            covered.add(ci)
            correct, _, _ = s.answer(ci, lvl)
            t.update_idx(ci, lvl, correct); s.learn(ci, lvl, correct)
        final_apr = float(np.mean(t.get_mastery_matrix()[:, 0]))
        aprs.append(final_apr); steps.append(step + 1); goals.append(final_apr >= cfg.BETA_APR)
    return {
        "mean_apr":   float(np.mean(aprs)),
        "std_apr":    float(np.std(aprs)),
        "goal_rate":  float(np.mean(goals)) * 100,
        "mean_steps": float(np.mean(steps)),
        "diff_dist":  diff_counts / max(diff_counts.sum(), 1),
    }


def _eppo_fn(ag):
    def fn(t, mastery, rng, cfg):
        with torch.no_grad():
            a, _, _, _ = ag.select_action(t.get_state_vector(), mastery)
        return a // cfg.BLOOM_LEVELS, a % cfg.BLOOM_LEVELS + 1
    return fn

def _random_fn(t, mastery, rng, cfg):
    return int(rng.integers(0, cfg.N_CONCEPTS)), int(rng.integers(1, cfg.BLOOM_LEVELS + 1))

def _greedy_fn(t, mastery, rng, cfg):
    ci  = int(np.argmin(mastery[:, 0]))
    cur = max((k for k in range(cfg.BLOOM_LEVELS) if mastery[ci, k] > 0.55), default=0)
    return ci, min(cur + 1, cfg.BLOOM_LEVELS - 1) + 1


eval_rng = np.random.default_rng(77)
agent.eval()

policies = {
    "EPPO":   _run_policy_eval(_eppo_fn(agent), cfg, sim_matrix, eval_rng),
    "Random": _run_policy_eval(_random_fn,       cfg, sim_matrix, eval_rng),
    "Greedy": _run_policy_eval(_greedy_fn,       cfg, sim_matrix, eval_rng),
}

# ── Print table ──
W = 10
print(f"\n{'Metric':<25} {'EPPO':>{W}} {'Random':>{W}} {'Greedy':>{W}}")
print("-" * 55)
for label, key, fmt in [
    ("Mean APR",   "mean_apr",   ".3f"),
    ("Std APR",    "std_apr",    ".3f"),
    ("Goal %",     "goal_rate",  ".1f"),
    ("Mean steps", "mean_steps", ".1f"),
]:
    row = f"{label:<25}"
    for p in ["EPPO", "Random", "Greedy"]:
        row += f" {policies[p][key]:>{W}{fmt}}"
    print(row)

e, r, g = policies["EPPO"], policies["Random"], policies["Greedy"]
print(f"\nvs Random: {'EPPO wins (+{:.3f})'.format(e['mean_apr']-r['mean_apr']) if e['mean_apr']>r['mean_apr'] else 'Random wins -- more training needed'}")
print(f"vs Greedy: {'EPPO wins' if e['mean_apr']>g['mean_apr'] or e['mean_steps']<g['mean_steps'] else 'Greedy wins -- EPPO not converged'}")

# ── Bar charts ──
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
labels = ["EPPO", "Random", "Greedy"]
colors = ["steelblue", "tomato", "seagreen"]

axes[0].bar(labels, [policies[p]["mean_apr"] for p in labels], color=colors)
axes[0].axhline(cfg.BETA_APR, ls="--", color="black", alpha=0.5)
axes[0].set_title("Mean final APR"); axes[0].set_ylim(0, 1)

axes[1].bar(labels, [policies[p]["goal_rate"] for p in labels], color=colors)
axes[1].set_title("Goal reached %"); axes[1].set_ylim(0, 100)

axes[2].bar(labels, [policies[p]["mean_steps"] for p in labels], color=colors)
axes[2].set_title("Mean steps to finish")

plt.tight_layout()
plt.savefig("/kaggle/working/policy_comparison.png", dpi=120)
plt.show()

# ── Bloom distribution ──
bloom_names = ["L1\nRecall","L2\nUnderstand","L3\nApply","L4\nAnalyse","L5\nEvaluate","L6\nCreate"]
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, (p, col) in zip(axes, zip(["EPPO","Random","Greedy"], colors)):
    ax.bar(range(cfg.BLOOM_LEVELS), policies[p]["diff_dist"] * 100, color=col, alpha=0.85)
    ax.set_title(f"{p} — Bloom distribution")
    ax.set_xticks(range(cfg.BLOOM_LEVELS)); ax.set_xticklabels(bloom_names, fontsize=8)
    ax.set_ylabel("% of actions")
plt.tight_layout()
plt.savefig("/kaggle/working/bloom_distribution.png", dpi=120)
plt.show()


## 14 · Demo session — step-by-step trace

Shows exactly what EPPO recommends at each step for one student, and how mastery evolves.  
Try `seed=0`, `seed=7`, or `seed=99` to see different student profiles.


In [ ]:
DEMO_SEED = 42

rng_demo = np.random.default_rng(DEMO_SEED)
tracker  = PFATracker(cfg.CONCEPTS, cfg, sim_matrix=sim_matrix)
student  = RealisticStudent.from_archetype(cfg.CONCEPTS, sim_matrix, rng_demo, cfg, "mixed")
agent.eval()

BLOOM_NAMES = ["Recall", "Understand", "Apply", "Analyse", "Evaluate", "Create"]

print("=" * 72)
print("  DEMO SESSION — step by step")
print("=" * 72)
print(f"  Goal: APR >= {cfg.BETA_APR}")
print()
print(f"  {'Step':>4}  {'Concept':<24} {'Level':<12} {'Mastery':>8} {'Answer':>8} {'APR':>6}")
print("  " + "-" * 68)

covered = set()
for step in range(cfg.MAX_STEPS):
    mastery = tracker.get_mastery_matrix()
    apr     = float(np.mean(mastery[:, 0]))
    if apr >= cfg.BETA_APR and len(covered) >= cfg.N_CONCEPTS:
        print(f"\n  Goal reached at step {step}!  Final APR = {apr:.3f}")
        break

    with torch.no_grad():
        action, _, _, _ = agent.select_action(tracker.get_state_vector(), mastery)
    ci  = action // cfg.BLOOM_LEVELS
    lvl = action  % cfg.BLOOM_LEVELS + 1

    concept = cfg.CONCEPTS[ci]
    m_val   = mastery[ci, lvl - 1]

    correct, true_p, p_know = student.answer(ci, lvl)
    tracker.update_idx(ci, lvl, correct)
    student.learn(ci, lvl, correct)
    covered.add(ci)

    ans = "CORRECT" if correct else "wrong  "
    print(f"  {step+1:>4}  {concept:<24} {BLOOM_NAMES[lvl-1]:<12} {m_val:>8.3f} {ans:>8} {apr:>6.3f}")

print()
mastery = tracker.get_mastery_matrix()
print(f"  {'Concept':<24} {'L1':>5} {'L2':>5} {'L3':>5} {'L4':>5} {'L5':>5} {'L6':>5}  Progress")
print("  " + "-" * 72)
for i, c in enumerate(cfg.CONCEPTS):
    vals = mastery[i]
    bar  = "█" * int(vals[0] * 10) + "░" * (10 - int(vals[0] * 10))
    print(f"  {c:<24} " + " ".join(f"{v:>5.2f}" for v in vals) + f"  {bar}")


## 15 · Readiness checklist

Go / no-go before connecting to the question generator or scaling to the full concept set.


In [ ]:
print("\n" + "=" * 60)
print("  READINESS CHECKLIST")
print("=" * 60)

sl        = slice(-200, None)
mean_apr  = np.mean(metrics["aprs"][sl])
goal_rate = np.mean(metrics["goals"][sl]) * 100
mean_steps= np.mean(metrics["steps"][sl])
ece       = val_results.get("ece", 1.0)
auc       = val_results.get("auc", 0.0)

e = policies["EPPO"]; r = policies["Random"]; g = policies["Greedy"]
beats_rnd = e["mean_apr"] > r["mean_apr"]
beats_grd = e["mean_apr"] > g["mean_apr"] or e["mean_steps"] < g["mean_steps"]

checks = [
    ("PFA ECE < 0.10",             ece < 0.10,         f"ECE={ece:.4f}"),
    ("PFA AUC > 0.65",             auc > 0.65,         f"AUC={auc:.3f}"),
    ("Final APR > 0.60",           mean_apr > 0.60,    f"APR={mean_apr:.3f}"),
    ("Goal rate > 40%",            goal_rate > 40,     f"{goal_rate:.1f}%"),
    ("Avg steps 10-30",            10 < mean_steps < 30, f"{mean_steps:.1f} steps"),
    ("EPPO beats Random",          beats_rnd,          f"EPPO={e['mean_apr']:.3f} vs Rnd={r['mean_apr']:.3f}"),
    ("EPPO competitive w/ Greedy", beats_grd,          f"EPPO={e['mean_apr']:.3f} vs Grd={g['mean_apr']:.3f}"),
]

all_pass = True
for label, ok, detail in checks:
    print(f"  [{'PASS' if ok else 'FAIL'}] {label:<35} {detail}")
    if not ok:
        all_pass = False

print()
if all_pass:
    print("  All checks passed — ready to scale to full concept set.")
    print("  Next steps:")
    print("    1. Replace CONCEPTS list with extracted domain concepts")
    print("    2. Connect output (concept, bloom_level) to question generator")
    print("    3. Collect real student interactions and retrain")
else:
    print("  Some checks failed:")
    if ece >= 0.10:
        print("    ECE high -> lower GAMMA_LEVEL or align BETA_LEVEL to IRT curve")
    if goal_rate <= 40:
        print("    Goal rate low -> increase N_EPISODES or lower BETA_APR")
    if mean_steps >= 30:
        print("    Sessions too long -> check MIN_COVERAGE or MAX_SAME_CONCEPT")
    if not beats_rnd:
        print("    EPPO not beating random -> increase N_EPISODES to 2500 or raise LR_ACTOR")

print("=" * 60)
